<a href="https://colab.research.google.com/github/anyuanay/INFO323/blob/main/INFO323_Lecture_week6_Intro_Spark_Programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="text-align:center"> INFO 323: Cloud Computing and Big Data</h1>
<h2 style="text-align:center"> College of Computing and Informatics</h2>
<h2 style="text-align:center">Drexel University</h2>

<h3 style="text-align:center"> Introduction to Spark Programming</h3>
<h3 style="text-align:center"> Yuan An, PhD</h3>
<h3 style="text-align:center">Associate Professor</h3>

# Cloud Computing Recap
- Typically, when we think of a “computer,” we think about one machine sitting on our desk at home or at work.
- There are some things that our computer is not powerful enough to perform. One particularly challenging area is data processing.
- Single machines do not have enough power and resources to perform computations on huge amounts of information (or the user probably does not have the time to wait for the computation to finish).
- A cluster, or group, of computers, pools the resources of many machines together, giving us the ability to use all the cumulative resources as if they were a single computer.


# Cloud Computing and Spark
- Now, a group of machines alone is not powerful, we need a framework to coordinate work across them.
- Spark does just that, managing and coordinating the execution of tasks on data across a cluster of computers.
- The cluster of machines that Spark will use to execute tasks is managed by a cluster manager like
 - * Spark’s standalone cluster manager,
 - * YARN,
 - * or Mesos.
- We then submit Spark Applications to these cluster managers, which will grant resources to our application so that we can complete our work.


# Check Spark Availability

In [ ]:
spark.version

'4.1.0'

# Test Spark

## DataFrames
Like Pandas, Spark also provides high-level APIs on structured data. The most common structured data format is DataFrame, which represents a table of data with rows and columns.

![](https://i.imgur.com/qvdB1rT.png)

### Load a CSV File as a Spark DataFrame
- In Databricks, create a new volume `info323` in Catalog under `workspace/default`.
- Download the file `2015-summary.csv` from the course shell.
- Upload the file to the volume `/Volume/workspace/default/info323`.

Spark reads in a DataFrame from a file. The DataFrame has a set of columns with an unspecified number of rows. The reason the number of rows is unspecified is because reading data is a transformation, and
is therefore a lazy operation. Spark peeked at only a couple of rows of data to try to guess what types
each column should be.

![](https://i.imgur.com/o1y1JzQ.png)

In [ ]:
# Uploaded to Catalog
file="/Volumes/workspace/default/info323/2015-summary.csv"

In [ ]:
flightdata = spark.read.option("inferSchema", "true").option("header", "true").csv(file)

### View the Head of the DataFrame

In [ ]:
flightdata.head(5)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count=344),
 Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count=62)]

### Print the Schema of the DataFrame

In [ ]:
flightdata.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



### Spark Transformations

Let us specify a transformation, where(). Nothing happens to the data when we call the transformation, because it’s just a transformation.


In [ ]:
# filter rows on condition - not evaluated yet
# this is a narrow transformation
flightdata.where(flightdata.DEST_COUNTRY_NAME == 'United States')

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: int]

In [ ]:
# show the results by action
flightdata.where(flightdata.DEST_COUNTRY_NAME == 'United States').show(3)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
+-----------------+-------------------+-----+
only showing top 3 rows


Let us specify a wide transformation, sort(). Again, nothing happens to the data when we call sort because it’s just a transformation. However, we can see that Spark is building up a plan for how it will execute this across the cluster by looking at the
explain plan. We can call explain on any DataFrame object to see the DataFrame’s lineage.

In [ ]:
# A wide transformation
flightdata.sort("count")

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: int]

In [ ]:
flightdata.sort("count").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [count#10971 ASC NULLS FIRST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7162]
               +- PhotonShuffleExchangeSink rangepartitioning(count#10971 ASC NULLS FIRST, 16)
                  +- PhotonRowToColumnar
                     +- FileScan csv [DEST_COUNTRY_NAME#10969,ORIGIN_COUNTRY_NAME#10970,count#10971] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/info323/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>


== Photon Explanation ==
The query is fully supported by Photon.


Next, we can specify an action to kick off this plan. However, before doing
that, we’re going to set a configuration. By default, when we perform a shuffle, Spark outputs 200
shuffle partitions. Let’s set this value to 5 to reduce the number of the output partitions from the
shuffle.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "5")

In [ ]:
# Actually execute it:
flightdata.sort("count").take(2)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', count=1)]

# Retrieval Practice 1

### DataFrame and SQL

We worked through a simple transformation in the previous example, let’s now work through a more
complex one and follow along in both DataFrames and SQL. Spark can run the same transformations,
regardless of the language, in the exact same way. You can express your business logic in SQL or
DataFrames (either in R, Python, Scala, or Java) and Spark will compile that logic down to an
underlying plan (that you can see in the explain plan) before actually executing your code. With Spark
SQL, you can register any DataFrame as a table or view (a temporary table) and query it using pure
SQL. There is no performance difference between writing SQL queries or writing DataFrame code,
they both “compile” to the same underlying plan that we specify in DataFrame code.
You can make any DataFrame into a table or view with one simple method call:

![](https://i.imgur.com/uFU3s01.png)

In [ ]:
# Register the DataFrame as a relational table view
flightdata.createOrReplaceTempView("flight_data_table")

We can query our data in SQL. To do so, we’ll use the spark.sql function (remember, spark is our SparkSession variable) that conveniently returns a new DataFrame. Although this might seem a bit circular in logic—that a SQL query against a DataFrame returns another DataFrame—it’s actually
quite powerful. This makes it possible for you to specify transformations in the manner most convenient to you at any given point in time and not sacrifice any efficiency to do so! To understand
that this is happening, let’s take a look at two explain plans:

In [ ]:
flightdata.groupBy("DEST_COUNTRY_NAME").count().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#10969], functions=[finalmerge_count(merge count#11002L) AS count(1)#11000L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7232]
               +- PhotonShuffleExchangeSink hashpartitioning(DEST_COUNTRY_NAME#10969, 5)
                  +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#10969], functions=[partial_count(1) AS count#11002L])
                     +- PhotonRowToColumnar
                        +- FileScan csv [DEST_COUNTRY_NAME#10969] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/info323/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>


== Photon Explanation ==
The query is fully supported by Photon.


In [ ]:
sqlWay = spark.sql("""
SELECT DEST_COUNTRY_NAME, count(1)
FROM flight_data_table
GROUP BY DEST_COUNTRY_NAME
""")

In [ ]:
sqlWay.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#10969], functions=[finalmerge_count(merge count#11013L) AS count(1)#11010L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7275]
               +- PhotonShuffleExchangeSink hashpartitioning(DEST_COUNTRY_NAME#10969, 5)
                  +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#10969], functions=[partial_count(1) AS count#11013L])
                     +- PhotonRowToColumnar
                        +- FileScan csv [DEST_COUNTRY_NAME#10969] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/default/info323/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>


== Photon Explanation ==
The query is fully supported by Photon.


Let’s pull out some interesting statistics from our data. One thing to understand is that DataFrames
(and SQL) in Spark already have a huge number of manipulations available. There are hundreds of
functions that you can use and import to help you resolve your big data problems faster. We will use
the max function, to establish the maximum number of flights to and from any given location. This just
scans each value in the relevant column in the DataFrame and checks whether it’s greater than the
previous values that have been seen. This is a transformation, because we are effectively filtering
down to one row. Let’s see what that looks like:

In [ ]:
spark.sql("SELECT max(count) FROM flight_data_table").take(1)

[Row(max(count)=370002)]

Let’s perform something a bit more
complicated and find the top five destination countries in the data. This is our first multitransformation
query, so we’ll take it step by step. Let’s begin with a fairly straightforward SQL
aggregation:

In [ ]:
maxsql = spark.sql("""
SELECT DEST_COUNTRY_NAME, sum(count) as destination_total
FROM flight_data_table
GROUP BY DEST_COUNTRY_NAME
ORDER BY sum(count) DESC
LIMIT 5
""")

In [ ]:
maxsql.show()

+-----------------+-----------------+
|DEST_COUNTRY_NAME|destination_total|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+



Now, let’s move to the DataFrame syntax that is semantically similar but slightly different in
implementation and ordering. But, as we mentioned, the underlying plans for both of them are the
same. Let’s run the queries and see their results as a sanity check:

In [ ]:
from pyspark.sql.functions import desc

In [ ]:
flightdata.groupBy("DEST_COUNTRY_NAME")\
.sum("count").withColumnRenamed("sum(count)", "destination_total")\
.sort(desc("destination_total")).limit(5).show()

+-----------------+-----------------+
|DEST_COUNTRY_NAME|destination_total|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+



The above sequence of transformations has 7 transformation steps: read->groupBy->sum->withColumnRenamed->sort->limit->collect

In [ ]:
flightdata.groupBy("DEST_COUNTRY_NAME")\
.sum("count").withColumnRenamed("sum(count)", "destination_total")\
.sort(desc("destination_total")).limit(5).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonTopK(sortOrder=[destination_total#11075L DESC NULLS LAST], partitionOrderCount=0)
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7641]
               +- PhotonShuffleExchangeSink SinglePartition
                  +- PhotonTopK(sortOrder=[destination_total#11075L DESC NULLS LAST], partitionOrderCount=0)
                     +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#10969], functions=[finalmerge_sum(merge sum#11077L) AS sum(count)#11073L])
                        +- PhotonShuffleExchangeSource
                           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7633]
                              +- PhotonShuffleExchangeSink hashpartitioning(DEST_COUNTRY_NAME#10969, 5)
                                 +- PhotonGroupingAgg(keys=[DEST_COUNTRY_NAME#10969], functions=[partial_sum(count#1

# Retrieval Practice 2

# What is Spark?

![](https://i.imgur.com/NQiRXn7.png)

- **De-facto standard unified analytics engine for big data processing**
- **Largest open-source project in data processing**
- **Technology created by the founders of Databricks**

# Features of Spark
- Spark implements a distributed, fault-tolerant, in-memory structure called a Resilient Distributed Dataset (RDD).
- Spark maximizes the use of memory across multiple machines, significantly improving overall performance.
- Spark’s reuse of these in-memory structures makes it well suited to iterative machine learning operations as well as interactive queries.


# Programming Interfaces to Spark
Spark provides native support for programming interfaces including the following:
- Scala
- Python (using Python’s functional programming operators)
- Java
- SQL
- R


# Why Spark?
- Expressive programming model
- In-memory processing
- Support for diverse workloads
- Interactive shell


# Spark Background
- An open source distributed data processing project started in 2009 by Matei Zaharia at the University of California, Berkeley, RAD Lab.
- As part of the Mesos research project, designed to look at an alternative resource scheduling and orchestration system to MapReduce.
- Became an alternative to using traditional MapReduce on Hadoop.


# Benefits of Spark
- Easy to use
- Unified
- Fast


# Use of Spark
- Extract-transform-load (ETL) operations
- Predictive analytics and machine learning
- Data access operations, such as SQL queries and visualizations
- Text mining and text processing
- Real-time event processing
- Graph applications
- Pattern recognition
- Recommendation engines


# Spark Programs
A program consists of a sequence of transformations followed by an action.
![](https://i.imgur.com/NLTWLo0.png)


# Transformations
- In Spark, the core data structures are immutable, meaning they cannot be changed after they’re created.
- This might seem like a strange concept at first: if we cannot change it, how are we supposed to use it?
- To “change” a DataFrame, we need to instruct Spark how we would like to modify it to do what we want.
- These instructions are called transformations.


# Narrow Transformation
Transformations consisting of narrow dependencies (we’ll call them narrow transformations) are
those for which each input partition will contribute to only one output partition.
# Wide Transformation
A wide dependency (or wide transformation) style transformation will have input partitions
contributing to many output partitions. You will often hear this referred to as a shuffle whereby Spark
will exchange partitions across the cluster. With narrow transformations, Spark will automatically
perform an operation called pipelining, meaning that if we specify multiple filters on DataFrames,
they’ll all be performed in-memory. The same cannot be said for shuffles. When we perform a shuffle,
Spark writes the results to disk.
![narrow vs wide](https://i.imgur.com/jJ4fypS.png)

# Lazy Evaluation
Lazy evaluation means that Spark will wait until the very last moment to execute the graph of
computation instructions.

In Spark, instead of modifying the data immediately when you express some
operation, you build up a plan of transformations that you would like to apply to your source data.

By waiting until the last minute to execute the code, Spark compiles this plan from your raw DataFrame
transformations to a streamlined physical plan that will run as efficiently as possible across the
cluster.

This provides immense benefits because Spark can optimize the entire data flow from end to
end.

An example of this is something called predicate pushdown on DataFrames. If we build a large
Spark job but specify a filter at the end that only requires us to fetch one row from our source data,
the most efficient way to execute this is to access the single record that we need. Spark will actually
optimize this for us by pushing the filter down automatically.


# Actions
- To trigger the computation, we run an action - instructs Spark to compute a result from a series of transformations.
- There are three kinds of actions:
- * Actions to view data in the console
- * Actions to collect data to native objects in the respective language
- * Actions to write to output data sources


# Spark Applications
Spark Applications consist of a driver process and a set of executor processes.
![](https://i.imgur.com/zF7Ngfc.png)

# Spark Executor
Each executor’s core gets a partition of data to work on
![](https://i.imgur.com/FZN5dCb.png)

# Spark Driver
- Each Spark driver creates one or more Spark jobs;
- Each Spark job creates one or more stages;
- Each Spark stage creates one or more tasks to be distributed to executors
![](https://i.imgur.com/lMTRd1c.png)

# Spark API

![](https://i.imgur.com/B93lBgz.png)

# Spark Input and Output Types
Although Spark is mostly used to process data in Hadoop, Spark can be used with a multitude of other source and target systems, including the following:
- Local or network filesystems
- Object storage such as Amazon S3 or Ceph
- Relational database systems
- NoSQL stores, including Cassandra, HBase, and others
- Messaging systems such as Kafka


# Something about Spark RDD
- Typically, we don't directly access the underlying Spark data structure which is Resilient Distributed Dataset (RDD).
- Spark uses RDD to distribute data across a cluster.
- RDDs are in memory, fast, and support fault tolerance.

Although RDDs were the primary data abstraction in early versions of Spark, the introduction of DataFrames and Datasets has provided more optimized and higher-level abstractions for most tasks. However, RDDs are still useful for cases where complete control over data handling and manipulation is required.


# Retrieval Practice 3